# Problema 3: Rutas de un vehículo autónomo

*Enunciado adaptado del problema 9 del Banco de Problemas de Aprendizaje por Refuerzo (MDT-Módulo 3, IIND-3518). Al final del cuaderno se explica qué se modificó.*

**Enunciado:** Un vehículo autónomo opera en una cuadrícula de $3 \times 5$. Las coordenadas son (fila, columna), con $(0,0)$ arriba a la izquierda. Parte de $S = (2,0)$ y debe hacer entregas en $G = (2,4)$. Las posiciones $(2,1)$, $(2,2)$ y $(2,3)$ son peligrosas.

| | 0 | 1 | 2 | 3 | 4 |
|---|---|---|---|---|---|
| **0** | | | | | |
| **1** | | | | | |
| **2** | S | ! | ! | ! | G |

Las acciones, indexadas 0, 1, 2 y 3, son arriba, derecha, abajo e izquierda. Un movimiento fuera de la cuadrícula deja al vehículo en el mismo lugar. Todo movimiento normal genera $-1$. Entrar en una posición peligrosa genera $-30$ y devuelve al vehículo a $S$. Llegar a $G$ genera $+10$ por la entrega y el vehículo vuelve a $S$. El vehículo opera indefinidamente y se usa $\gamma = 0{,}95$.

- (a) Formule el problema para Q-learning: estados, acciones factibles y qué observa el algoritmo después de cada decisión.
- (b) Entrene Q-learning con $\alpha = 0{,}1$, $\varepsilon = 0{,}1$, $2 \times 10^5$ pasos y semilla 2026. Reporte la trayectoria greedy desde $S$ hasta $G$ y su recompensa.
- (c) Calcule la política óptima de forma exacta con *value iteration*. Compare la ganancia máxima de cada estado con la ganancia de seguir la política aprendida en (b).
- (d) Repita el entrenamiento con $\varepsilon = 0{,}3$ y la misma semilla. Compare la trayectoria greedy y la ganancia de la política aprendida con las de $\varepsilon = 0{,}1$.

---
## Preparación: importar la librería

In [1]:
import sys
import os.path
import numpy as np
# context for the examples
sys.path.insert(0, os.path.abspath(os.path.join(os.getcwd(), '..', '..', '..')))
from jmarkov.mdp.qlearning import qlearning
from jmarkov.mdp.dtmdp import dtmdp

---
## Solución (a)

**Aquí se usa la librería:** el problema se describe con las tres funciones que recibe la clase `qlearning`.

- **Estados:** la posición del vehículo, (fila, columna). Las posiciones peligrosas y $G$ nunca son el estado en que queda el vehículo, porque al entrar en ellas vuelve a $S$. Quedan 11 estados. La función `initial_state` devuelve $S = (2,0)$.
- **Acciones factibles:** las cuatro direcciones en todos los estados. Lo expresa la función `feasible_actions`.
- **Qué observa el algoritmo:** la función `step` mueve el vehículo y devuelve **(posición siguiente, recompensa)**: $-1$ por un movimiento normal, $-30$ y vuelta a $S$ por entrar al peligro, y $+10$ y vuelta a $S$ por llegar a $G$.

In [2]:
GAMMA = 0.95
S = (2, 0)
G = (2, 4)
PELIGRO = {(2, 1), (2, 2), (2, 3)}
MOVIMIENTO = [(-1, 0), (0, 1), (1, 0), (0, -1)]   # 0 arriba, 1 derecha, 2 abajo, 3 izquierda
NOMBRE = ["arriba", "derecha", "abajo", "izquierda"]


def initial_state(rng):
    return S


def feasible_actions(state):
    return (0, 1, 2, 3)


def step(state, action, rng):
    fila = state[0] + MOVIMIENTO[action][0]
    columna = state[1] + MOVIMIENTO[action][1]
    if not (0 <= fila < 3 and 0 <= columna < 5):   # fuera de la cuadrícula: se queda
        fila, columna = state
    if (fila, columna) in PELIGRO:                 # peligro: -30 y vuelve a S
        return S, -30.0
    elif (fila, columna) == G:                       # entrega: +10 y vuelve a S
        return S, 10.0
    return (fila, columna), -1.0                   # movimiento normal


ESTADOS = [(f, c) for f in range(3) for c in range(5) if (f, c) != G and (f, c) not in PELIGRO]
print(len(ESTADOS), "estados:", ESTADOS)

11 estados: [(0, 0), (0, 1), (0, 2), (0, 3), (0, 4), (1, 0), (1, 1), (1, 2), (1, 3), (1, 4), (2, 0)]


---
## Solución (b)

**Aquí se usa la librería:** `model.solve(...)` entrena Q-learning durante $2 \times 10^5$ pasos, siguiendo una sola trayectoria desde $S$ y eligiendo las acciones con $\varepsilon$-greedy. Devuelve la política greedy, con la que luego se recorre el camino desde $S$ hasta $G$ sin exploración.

In [3]:
def trayectoria_greedy(politica):
    """Recorre la cuadrícula desde S con la política dada, hasta llegar a G."""
    estado, camino, total = S, [S], 0.0
    for _ in range(50):
        siguiente, recompensa = step(estado, politica[estado], None)
        total += recompensa
        if recompensa == 10.0:                 # llegó a G
            camino.append(G)
            break
        camino.append(siguiente)
        estado = siguiente
    return camino, total


model = qlearning(initial_state, feasible_actions, step, discount_factor=GAMMA)  
valores_01, politica_01 = model.solve(n_steps=200_000, alpha=0.1, epsilon=0.1, minimize=False, seed=2026)        

camino, total = trayectoria_greedy(politica_01)
print("Trayectoria greedy:", " -> ".join(str(p) for p in camino))
print("Movimientos:", len(camino) - 1, "| Recompensa del recorrido:", total)

Trayectoria greedy: (2, 0) -> (1, 0) -> (1, 1) -> (1, 2) -> (1, 3) -> (1, 4) -> (2, 4)
Movimientos: 6 | Recompensa del recorrido: 5.0


La trayectoria evita las posiciones peligrosas: sube a la fila 1, la recorre hasta la columna 4 y baja a $G$. Son seis movimientos: cinco de $-1$ y la entrega de $+10$, es decir, una recompensa de 5 por recorrido. Es la misma ruta segura de la solución del problema original.

---
## Solución (c)

**Aquí se usa jMarkov, pero no la clase de Q-learning:** la solución exacta se obtiene con la clase `dtmdp`. Como el movimiento es determinístico, cada matriz de transición tiene un 1 en la posición a la que lleva cada acción, y la matriz de recompensas tiene la recompensa de cada movimiento.

In [4]:
indice = {s: i for i, s in enumerate(ESTADOS)}
matrices = {a: np.zeros((len(ESTADOS), len(ESTADOS))) for a in range(4)}
recompensas = np.zeros((len(ESTADOS), 4))
for s in ESTADOS:
    for a in range(4):
        siguiente, r = step(s, a, None)
        matrices[a][indice[s], indice[siguiente]] = 1.0
        recompensas[indice[s], a] = r

etiquetas = np.array([str(s) for s in ESTADOS])
mdp = dtmdp(etiquetas, np.arange(4), matrices, recompensas, GAMMA)                              
valores_exactos, politica_exacta = mdp.solve(1e-12, minimize=False, method="value_iteration")   

# ganancia con mdp y con qlearning
def ganancia(politica):
    """Ganancia exacta de seguir una política: resuelve (I - gamma P_pi) V = r_pi."""
    P_pi = np.array([matrices[politica[s]][indice[s]] for s in ESTADOS])
    r_pi = np.array([recompensas[indice[s], politica[s]] for s in ESTADOS])
    return np.linalg.solve(np.eye(len(ESTADOS)) - GAMMA * P_pi, r_pi)


ganancia_01 = ganancia(politica_01)
print(f"{'Estado':>8} {'Acción exacta':>14} {'Acción Q-learning':>18} {'Ganancia máxima':>16} {'Ganancia Q-learning':>20}")
for i, s in enumerate(ESTADOS):
    print(f"{str(s):>8} {NOMBRE[int(politica_exacta[str(s)])]:>14} {NOMBRE[politica_01[s]]:>18} "
          f"{valores_exactos[i]:>16.3f} {ganancia_01[i]:>20.3f}")

  Estado  Acción exacta  Acción Q-learning  Ganancia máxima  Ganancia Q-learning
  (0, 0)        derecha              abajo           12.130               12.130
  (0, 1)        derecha              abajo           13.821               13.821
  (0, 2)        derecha              abajo           15.601               15.601
  (0, 3)        derecha              abajo           17.475               17.475
  (0, 4)          abajo              abajo           19.448               19.448
  (1, 0)        derecha            derecha           13.821               13.821
  (1, 1)        derecha            derecha           15.601               15.601
  (1, 2)        derecha            derecha           17.475               17.475
  (1, 3)        derecha            derecha           19.448               19.448
  (1, 4)          abajo              abajo           21.524               21.524
  (2, 0)         arriba             arriba           12.130               12.130


La ganancia de la política aprendida es igual a la máxima en todos los estados, así que la política aprendida es óptima. En algunas posiciones de la fila 0 las dos políticas eligen acciones distintas (por ejemplo, "abajo" frente a "derecha"): son **empates**, porque desde esas posiciones los dos movimientos llevan a $G$ en el mismo número de pasos y dan la misma ganancia.

---
## Solución (d)

**Aquí se usa la librería:** se crea un modelo nuevo y se entrena con $\varepsilon = 0{,}3$ y la misma semilla. `solve` empieza siempre con la tabla $Q$ en cero, así que el entrenamiento es independiente del anterior.

In [5]:
model_03 = qlearning(initial_state, feasible_actions, step, discount_factor=GAMMA)   
valores_03, politica_03 = model_03.solve(
    n_steps=200_000, alpha=0.1, epsilon=0.3, minimize=False, seed=2026)             

camino_03, total_03 = trayectoria_greedy(politica_03)
ganancia_03 = ganancia(politica_03)

print("epsilon = 0.1:", " -> ".join(str(p) for p in camino), f"| recompensa {total}")
print("epsilon = 0.3:", " -> ".join(str(p) for p in camino_03), f"| recompensa {total_03}")
print(f"\nGanancia desde S: máxima {valores_exactos[indice[S]]:.3f} | "
      f"epsilon 0.1: {ganancia_01[indice[S]]:.3f} | epsilon 0.3: {ganancia_03[indice[S]]:.3f}")

epsilon = 0.1: (2, 0) -> (1, 0) -> (1, 1) -> (1, 2) -> (1, 3) -> (1, 4) -> (2, 4) | recompensa 5.0
epsilon = 0.3: (2, 0) -> (1, 0) -> (1, 1) -> (1, 2) -> (1, 3) -> (1, 4) -> (2, 4) | recompensa 5.0

Ganancia desde S: máxima 12.130 | epsilon 0.1: 12.130 | epsilon 0.3: 12.130


Con $\varepsilon = 0{,}3$ la trayectoria greedy y la ganancia son las mismas que con $\varepsilon = 0{,}1$: ambas políticas son óptimas. Con este número de pasos, aumentar la exploración no cambió la política aprendida.

---
## Nota sobre la adaptación del problema original de la base de problemas

En el problema original cada recorrido es un episodio que termina al llegar a $G$, y se usa $\gamma = 1$. La librería resuelve problemas continuos con descuento ($\gamma < 1$) en una sola trayectoria. Por eso se hicieron dos cambios:

- Al llegar a $G$, el vehículo recibe $+10$ por la entrega y vuelve a $S$ para iniciar otro recorrido. Sin esta recompensa, todos los movimientos valdrían $-1$ para siempre y al vehículo le daría igual llegar o no a $G$.
- Se usa $\gamma = 0{,}95$ en lugar de $\gamma = 1$.

La cuadrícula, las acciones, la penalización por peligro y el costo de cada movimiento son los del enunciado original.